In [1]:
import pandas as pd
import numpy as np
import os
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset, DataLoader
import torch
from transformers import AutoTokenizer, AutoModel
import torch.nn as nn
from torch.optim import AdamW
from torch.nn import BCEWithLogitsLoss
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, precision_score, recall_score, confusion_matrix, classification_report, roc_curve, auc
from peft import LoraConfig, get_peft_model  # For efficient fine-tuning

e:\ML\BioActivity\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_excel('bioactivity_dataset_cleaned_new_reg.xlsx')
df.head(3)

,canonical_smiles,MW,LogP,NumHDonors,pIC50,TPSA,NumRotatableBonds,FractionCSP3,RingCount
0,O=C(CCCCCC(NC(=O)OCc1ccccc1)C(=O)Nc1cccc2cccnc...,464.522,3.9242,4,8.6,129.65,11,0.280000,3
1,O=C(CCCCCC(C(=O)Nc1ccc2ncccc2c1)C(=O)Nc1ccc2nc...,485.544,4.4323,4,9.0,133.31,10,0.222222,4
2,O=C(/C=C/c1cccc(C(C(=O)Nc2ccccc2)C(=O)Nc2ccccc...,415.449,3.5662,4,9.0,107.53,7,0.041667,3


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6341 entries, 0 to 6340
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   canonical_smiles   6341 non-null   object 
 1   MW                 6341 non-null   float64
 2   LogP               6341 non-null   float64
 3   NumHDonors         6341 non-null   int64  
 4   pIC50              6341 non-null   float64
 5   TPSA               6341 non-null   float64
 6   NumRotatableBonds  6341 non-null   int64  
 7   FractionCSP3       6341 non-null   float64
 8   RingCount          6341 non-null   int64  
dtypes: float64(5), int64(3), object(1)
memory usage: 446.0+ KB


In [4]:
df.iloc[0]['canonical_smiles']

'O=C(CCCCCC(NC(=O)OCc1ccccc1)C(=O)Nc1cccc2cccnc12)NO'

In [5]:
import optuna
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs
from rdkit.Chem import rdFingerprintGenerator
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, precision_recall_curve
import numpy as np
import random
import warnings
warnings.filterwarnings("ignore")

# -------------------------------
# CONFIG (Fixed)
# -------------------------------
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MAX_LEN = 512
BATCH_SIZE = 32
ECFP_BITS = 1024
ECFP_RADIUS = 2
AUGMENT_TRAIN = False


train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

scaler=StandardScaler().fit(train_df[['MW', 'LogP', 'NumHDonors', 'TPSA', 'NumRotatableBonds', 'FractionCSP3', 'RingCount']])

# -------------------------------
# SMILES Augmentation
# -------------------------------
def random_smiles(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if not mol:
        return smiles
    return Chem.MolToSmiles(mol, doRandom=True, canonical=False)

# -------------------------------
# ECFP Generator
# -------------------------------
morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=ECFP_RADIUS, fpSize=ECFP_BITS)
def get_ecfp(smiles, radius=ECFP_RADIUS, n_bits=ECFP_BITS):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return np.zeros(n_bits, dtype=np.float32)
    fp = morgan_gen.GetFingerprint(mol)
    arr = np.zeros((n_bits,), dtype=np.float32)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

# -------------------------------
# Dataset
# -------------------------------

tokenizer = AutoTokenizer.from_pretrained('DeepChem/ChemBERTa-77M-MTR', trust_remote_code=True)

class BioActivityDataset(Dataset):
    def __init__(self, df, tokenizer, scaler=None, augment=False):
        self.smiles = df['canonical_smiles'].values.tolist()
        self.desc = df[['MW', 'LogP', 'NumHDonors', 'TPSA', 'NumRotatableBonds', 'FractionCSP3', 'RingCount']].values.astype(np.float32)
        self.labels = df['pIC50'].values.astype(np.float32)
        self.augment = augment
        
        if scaler is None:
            self.scaler = StandardScaler().fit(self.desc)
        else:
            self.scaler = scaler
        self.desc = self.scaler.transform(self.desc)
        
        print("Computing ECFP...")
        self.ecfp = np.stack([get_ecfp(s) for s in self.smiles])

        print("Tokenizing SMILES...")
        self.encodings = tokenizer(
            self.smiles,
            truncation=True,
            padding='max_length',
            max_length=512,
            return_tensors='pt'
        )

    def __len__(self):
        return len(self.smiles)

    def __getitem__(self, idx):
        return {
            'input_ids': self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'ecfp': torch.tensor(self.ecfp[idx]),
            'descriptors': torch.tensor(self.desc[idx]),
            'labels': torch.tensor(self.labels[idx])
        }

# -------------------------------
# Collate Function
# -------------------------------
def collate_fn(batch):
    return {
        'input_ids': torch.stack([b['input_ids'] for b in batch]),
        'attention_mask': torch.stack([b['attention_mask'] for b in batch]),
        'descriptors': torch.stack([b['descriptors'] for b in batch]),
        'ecfp': torch.stack([b['ecfp'] for b in batch]),
        'labels': torch.tensor([b['labels'] for b in batch], dtype=torch.float)
    }



#### ECFP + BERT embedding based model

In [6]:
# -------------------------------
# Model
# -------------------------------
class FineTunedBERTaECFP(nn.Module):
    def __init__(self, ecfp_bits=ECFP_BITS, unfreeze_layers=12, num_heads=8, 
                 dropout=0.3, hidden_dim=768, num_classifier_layers=4, 
                 fusion_type='gated'):
        super().__init__()
        self.bert = AutoModel.from_pretrained("DeepChem/ChemBERTa-77M-MTR", trust_remote_code=True)
        self.bert_dim = self.bert.config.hidden_size

        # Unfreeze last N layers
        for p in self.bert.parameters():
            p.requires_grad = False
        for layer in self.bert.encoder.layer[-unfreeze_layers:]:
            for p in layer.parameters():
                p.requires_grad = True

        # Project ECFP and descriptors to 768
        self.ecfp_proj = nn.Sequential(
            nn.Linear(ecfp_bits, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(512, self.bert_dim),
            nn.LayerNorm(self.bert_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        self.desc_proj = nn.Sequential(
            nn.Linear(7, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(256, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(512, self.bert_dim),
            nn.LayerNorm(self.bert_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        # Attention
        self.attn = nn.MultiheadAttention(self.bert_dim, num_heads=num_heads, batch_first=True, dropout=dropout*0.5)
        self.attn_norm = nn.LayerNorm(self.bert_dim)

        self.fusion_type = fusion_type
        if fusion_type == 'gated':
            self.gate = nn.Sequential(
                nn.Linear(self.bert_dim * 3, self.bert_dim),
                nn.Tanh()
            )
            self.fusion_proj = nn.Linear(self.bert_dim * 3, self.bert_dim)
        elif fusion_type == 'bilinear':
            self.bilinear = nn.Bilinear(self.bert_dim, self.bert_dim, self.bert_dim)

        # Classifier
        classifier_layers = []
        in_dim = self.bert_dim * 3 if fusion_type == 'concat' else self.bert_dim
        
        for i in range(num_classifier_layers):
            out_dim = hidden_dim // (2 ** i)
            classifier_layers.extend([
                nn.Linear(in_dim, out_dim),
                nn.LayerNorm(out_dim),
                nn.GELU(),
                nn.Dropout(dropout if i < num_classifier_layers - 1 else dropout * 0.5)
            ])
            in_dim = out_dim
        
        classifier_layers.append(nn.Linear(in_dim, 1))
        self.classifier = nn.Sequential(*classifier_layers)



    def forward(self, input_ids, attention_mask, ecfp, descriptors):
        # SMILES
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        
        cls_ = out.last_hidden_state[:,0]
        mean_pool = out.last_hidden_state.mean(dim=1)  # [B,768]
        max_pool = out.last_hidden_state.max(dim=1)[0]
        smiles = (cls_ + mean_pool + max_pool) / 3

        # ECFP + Desc
        ecfp_emb = self.ecfp_proj(ecfp)             # [B,768]
        desc_emb = self.desc_proj(descriptors)      # [B,768]

        # Attention
        q = torch.stack([ecfp_emb, desc_emb], dim=1)  # [B,2,768]
        kv = smiles.unsqueeze(1)                       # [B,1,768]
        attn_out, attn_weights = self.attn(q, kv, kv)             # [B,2,768]

        ecfp_attn = self.attn_norm(attn_out[:,0] + ecfp_emb)    # [B,768]
        desc_attn = self.attn_norm(attn_out[:,1] + desc_emb)   # [B,768]

        # Fuse
        if self.fusion_type == 'concat':
            fused = torch.cat([smiles, ecfp_attn, desc_attn], dim=1)
        elif self.fusion_type == 'gated':
            concat = torch.cat([smiles, ecfp_attn, desc_attn], dim=1)
            gate = self.gate(concat)
            fused = gate * self.fusion_proj(concat)
        else:  # bilinear
            fused = self.bilinear(smiles, ecfp_attn + desc_attn)
        
        return self.classifier(fused).squeeze(-1)

model = FineTunedBERTaECFP(unfreeze_layers=12, num_heads=8)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

Some weights of RobertaModel were not initialized from the model checkpoint at DeepChem/ChemBERTa-77M-MTR and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


FineTunedBERTaECFP(
  (bert): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(600, 384, padding_idx=1)
      (position_embeddings): Embedding(515, 384, padding_idx=1)
      (token_type_embeddings): Embedding(1, 384)
      (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.144, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-2): 3 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=384, out_features=384, bias=True)
              (key): Linear(in_features=384, out_features=384, bias=True)
              (value): Linear(in_features=384, out_features=384, bias=True)
              (dropout): Dropout(p=0.109, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=384, out_features=384, bias=True)
              (LayerNorm):

In [6]:
DEVICE

device(type='cuda')

#### Hyperparameter optimization

In [ ]:
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler
import torch.optim as optim_extra
import json
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import pearsonr

OPTUNA_CONFIG = {
    'n_trials': 50,
    'timeout': 3600 * 6,
    'study_name': 'bioactivity_regression',  # CHANGED
    'direction': 'minimize',  # CHANGED: minimize RMSE
    'seed': 42
}

# REMOVE FocalLoss class - not needed for regression


def simple_objective(trial):
    """
    Hyperparameter search for REGRESSION
    """
    
    # ========================================================================
    # 1. CORE HYPERPARAMETERS (Same)
    # ========================================================================
    
    unfreeze_layers = trial.suggest_int('unfreeze_layers', 4, 12)
    num_heads = trial.suggest_categorical('num_heads', [4, 8, 12, 16])
    hidden_dim = trial.suggest_categorical('hidden_dim', [256, 512, 768])
    dropout = trial.suggest_float('dropout', 0.1, 0.6, step=0.1)
    num_classifier_layers = trial.suggest_int('num_classifier_layers', 3, 5)
    fusion_type = trial.suggest_categorical('fusion_type', ['concat', 'gated', 'bilinear'])
    
    batch_size = trial.suggest_categorical('batch_size', [16, 32, 64])
    learning_rate = trial.suggest_float('learning_rate', 5e-7, 1e-5, log=True)
    weight_decay = trial.suggest_float('weight_decay', 1e-5, 1e-3, log=True)
    grad_accum_steps = trial.suggest_categorical('grad_accum_steps', [1, 2, 4, 8])
    
    optimizer_name = trial.suggest_categorical('optimizer', ['Adam', 'AdamW', 'RMSprop', 'RAdam'])
    scheduler_name = trial.suggest_categorical('scheduler', ['CosineAnnealingLR', 'ReduceLROnPlateau', 'Lambda', 'OneCycleLR', 'CosineAnnealingWarmRestarts'])

    # ========================================================================
    # 2. BUILD MODEL
    # ========================================================================
    
    try:
        model = FineTunedBERTaECFP(
            unfreeze_layers=unfreeze_layers,
            num_heads=num_heads,
            dropout=dropout,
            hidden_dim=hidden_dim,
            num_classifier_layers=num_classifier_layers,
            fusion_type=fusion_type
        ).to(DEVICE)
    except Exception as e:
        print(f"Model creation failed: {e}")
        raise optuna.exceptions.TrialPruned()
    
    # ========================================================================
    # 3. DATA LOADERS (Same)
    # ========================================================================
    
    print("Preparing datasets...")
    train_ds = BioActivityDataset(train_df, tokenizer, scaler=scaler)
    val_ds = BioActivityDataset(val_df, tokenizer, scaler=train_ds.scaler)
    print("Datasets ready.")
    
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_ds, batch_size=batch_size*2, shuffle=False, collate_fn=collate_fn)
    
    # ========================================================================
    # 4. LOSS & OPTIMIZER - CHANGED FOR REGRESSION
    # ========================================================================
    
    # CHANGED: Use MSE Loss for regression
    criterion = nn.MSELoss()
    # Alternative: nn.L1Loss() or nn.SmoothL1Loss()
    
    # Optimizer (same)
    if optimizer_name == 'Adam':
        optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    elif optimizer_name == 'AdamW':
        optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    elif optimizer_name == 'RMSprop':
        optimizer = torch.optim.RMSprop(model.parameters(), lr=learning_rate, weight_decay=weight_decay, momentum=0.9)
    else:
        optimizer = optim_extra.RAdam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    # Scheduler
    if scheduler_name == 'CosineAnnealingLR':
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)
    elif scheduler_name == 'CosineAnnealingWarmRestarts':
        scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=1e-7)
    elif scheduler_name == 'ReduceLROnPlateau':
        # CHANGED: mode='min' for regression
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
    elif scheduler_name == 'OneCycleLR':
        total_steps = len(train_loader) * 50
        scheduler = optim.lr_scheduler.OneCycleLR(
            optimizer, max_lr=learning_rate, total_steps=total_steps,
            pct_start=0.1, anneal_strategy='cos'
        )
    else:
        total_steps = len(train_loader) * 50
        warmup_steps = int(0.1 * total_steps)
        scheduler = optim.lr_scheduler.LambdaLR(
            optimizer,
            lr_lambda=lambda step: min(step/warmup_steps, 1) * (0.5 * (1 + np.cos(np.pi * step / total_steps)))
        )

    # ========================================================================
    # 5. TRAINING LOOP - CHANGED FOR REGRESSION
    # ========================================================================
    
    best_rmse = float('inf')  # CHANGED: track RMSE instead of AUC
    patience = 7
    patience_counter = 0
    MAX_EPOCHS = 50
    
    print("Starting training...")
    for epoch in range(MAX_EPOCHS):
        # ----------------------------------------------------------------
        # TRAINING
        # ----------------------------------------------------------------
        model.train()
        train_loss = 0.0

        for batch_idx, batch in enumerate(train_loader):
            try:
                input_ids = batch['input_ids'].to(DEVICE)
                attention_mask = batch['attention_mask'].to(DEVICE)
                ecfp = batch['ecfp'].to(DEVICE)
                descriptors = batch['descriptors'].to(DEVICE)
                labels = batch['labels'].float().to(DEVICE)

                logits = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    ecfp=ecfp,
                    descriptors=descriptors
                )

                # CHANGED: Direct MSE loss, no sigmoid
                loss = criterion(logits, labels) / grad_accum_steps
                loss.backward()
                
                if (batch_idx + 1) % grad_accum_steps == 0:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    optimizer.step()
                    optimizer.zero_grad()
                
                train_loss += loss.item() * grad_accum_steps

                if scheduler_name in ['Lambda', 'OneCycleLR']:
                    scheduler.step()
            
            except RuntimeError as e:
                if "out of memory" in str(e):
                    torch.cuda.empty_cache()
                    continue
                else:
                    raise e
        
        if scheduler_name == 'ReduceLROnPlateau':
            scheduler.step(train_loss / len(train_loader))  # Step with loss
        elif scheduler_name == 'CosineAnnealingLR':
            scheduler.step()
            
        # ----------------------------------------------------------------
        # VALIDATION - CHANGED FOR REGRESSION
        # ----------------------------------------------------------------
        model.eval()
        val_preds = []  # CHANGED: raw predictions
        val_labels = []
        
        with torch.no_grad():
            for batch in val_loader:
                try:
                    logits = model(
                        input_ids=batch['input_ids'].to(DEVICE),
                        attention_mask=batch['attention_mask'].to(DEVICE),
                        ecfp=batch['ecfp'].to(DEVICE),
                        descriptors=batch['descriptors'].to(DEVICE)
                    )
                    
                    # CHANGED: No sigmoid, use raw predictions
                    val_preds.extend(logits.cpu().numpy())
                    val_labels.extend(batch['labels'].cpu().numpy())
                
                except RuntimeError as e:
                    torch.cuda.empty_cache()
                    continue
        
        # CHANGED: Calculate regression metrics
        if len(val_preds) == 0:
            raise optuna.exceptions.TrialPruned()
        
        val_preds = np.array(val_preds)
        val_labels = np.array(val_labels)
        
        val_rmse = np.sqrt(mean_squared_error(val_labels, val_preds))
        val_mae = mean_absolute_error(val_labels, val_preds)
        val_r2 = r2_score(val_labels, val_preds)
        
        # Report RMSE to Optuna
        trial.report(val_rmse, epoch)
        
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()
        
        # Print progress
        if epoch % 5 == 0:
            print(f"Trial {trial.number} | Epoch {epoch+1:02d} | "
                  f"RMSE: {val_rmse:.4f} | MAE: {val_mae:.4f} | R²: {val_r2:.4f}")
        
        # Early stopping on RMSE
        if val_rmse < best_rmse - 1e-5:
            best_rmse = val_rmse
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break
    
    del model
    torch.cuda.empty_cache()
    
    return best_rmse  # CHANGED: return RMSE


# ============================================================================
# RUN OPTIMIZATION - CHANGED
# ============================================================================

def run_simple_optimization():
    """
    Run regression hyperparameter optimization
    """
    
    print("="*80)
    print("REGRESSION HYPERPARAMETER OPTIMIZATION")  # CHANGED
    print("="*80)
    print(f"Device: {DEVICE}")
    print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
    print(f"Trials: {OPTUNA_CONFIG['n_trials']}")
    print("="*80)
    
    study = optuna.create_study(
        study_name=OPTUNA_CONFIG['study_name'],
        direction=OPTUNA_CONFIG['direction'],  # 'minimize'
        sampler=TPESampler(seed=OPTUNA_CONFIG['seed']),
        pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=5)
    )
    
    study.optimize(
        simple_objective,
        n_trials=OPTUNA_CONFIG['n_trials'],
        timeout=OPTUNA_CONFIG['timeout'],
        show_progress_bar=True
    )
    
    # ========================================================================
    # RESULTS - CHANGED
    # ========================================================================
    
    print("\n" + "="*80)
    print("OPTIMIZATION COMPLETE")
    print("="*80)
    
    print(f"\n🏆 Best Trial: #{study.best_trial.number}")
    print(f"🎯 Best Validation RMSE: {study.best_value:.4f}")  # CHANGED
    
    print("\n📊 Best Hyperparameters:")
    print("-" * 80)
    for key, value in study.best_params.items():
        print(f"  {key:20s}: {value}")
    
    results = {
        'best_value': study.best_value,
        'best_params': study.best_params,
        'n_trials': len(study.trials)
    }
     
    with open(f'optuna_results_regression.json', 'w') as f:  # CHANGED filename
        json.dump(results, f, indent=4)
    
    print(f"\n✅ Results saved to: optuna_results_regression.json")
    
    completed = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
    pruned = [t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED]
    
    print("\n" + "="*80)
    print("TRIAL STATISTICS")
    print("="*80)
    print(f"Total trials: {len(study.trials)}")
    print(f"Completed: {len(completed)}")
    print(f"Pruned: {len(pruned)}")
    
    if completed:
        values = [t.value for t in completed]
        print(f"\nRMSE Statistics:")  # CHANGED
        print(f"  Best:   {min(values):.4f}")  # CHANGED: min instead of max
        print(f"  Mean:   {np.mean(values):.4f}")
        print(f"  Median: {np.median(values):.4f}")
        print(f"  Std:    {np.std(values):.4f}")
    
    return study

# Run optimization
study = run_simple_optimization()

### Training

In [7]:
# best_params = {
#     'unfreeze_layers'     : 8,
#   'num_heads'           : 4,
#   'hidden_dim'          : 768,
#   'dropout'             : 0.30000000000000004,
#   'batch_size'          : 16,
#   'learning_rate'       : 2.209652614551385e-05,
#   'weight_decay'        : 2.6471141828218167e-05,
#   'optimizer'           : 'Adam',
#   'scheduler'          : 'CosineAnnealingLR'
# }
# best_params

# best_params = {
#   'unfreeze_layers'     : 10,
#   'num_heads'           : 8,
#   'hidden_dim'          : 256,
#   'dropout'             : 0.2,
#   'batch_size'          : 64,
#   'learning_rate'       : 1.88414769215451e-05,
#   'weight_decay'        : 0.0045881565491609705,
#   'optimizer'           : 'Adam',
#   'scheduler'          : 'OneCycleLR'
# }

# best_params = {
#   'unfreeze_layers'     : 6,
#   'num_heads'           : 12,
#   'hidden_dim'          : 512,
#   'dropout'             : 0.30000000000000004,
#   'batch_size'          : 32,
#   'learning_rate'       : 9.863160901835706e-05,
#   'weight_decay'        : 0.0015656384285147025,
#   'optimizer'           : 'RAdam',
#   'scheduler'          : 'ReduceLROnPlateau'
# }

# {'unfreeze_layers': 17,
#  'num_heads': 12,
#  'hidden_dim': 512,
#  'dropout': 0.2,
#  'num_classifier_layers': 4,
#  'fusion_type': 'gated',
#  'batch_size': 32,
#  'learning_rate': 1.0429290698774817e-05,
#  'weight_decay': 0.00017783276419277671,
#  'grad_accum_steps': 1,
#  'optimizer': 'Adam',
#  'scheduler': 'ReduceLROnPlateau'}

best_params = {
  'unfreeze_layers'     : 5,
  'num_heads'           : 12,
  'hidden_dim'          : 768,
  'dropout'             : 0.30000000000000004,
  'num_classifier_layers': 3,
  'fusion_type'         : 'bilinear',
  'batch_size'          : 32,
  'learning_rate'       : 2.3746198818402034e-05,
  'weight_decay'        : 0.00012399967836846095,
  'grad_accum_steps'    : 2,
  'optimizer'           : 'AdamW',
  'scheduler'          : 'CosineAnnealingLR'
}
best_params

{'unfreeze_layers': 5,
 'num_heads': 12,
 'hidden_dim': 768,
 'dropout': 0.30000000000000004,
 'num_classifier_layers': 3,
 'fusion_type': 'bilinear',
 'batch_size': 32,
 'learning_rate': 2.3746198818402035e-05,
 'weight_decay': 0.00012399967836846095,
 'grad_accum_steps': 2,
 'optimizer': 'AdamW',
 'scheduler': 'CosineAnnealingLR'}

In [8]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from torch.amp import autocast, GradScaler

def retrain_best_model(best_params, epochs=50):
    """
    Retrain final model with best hyperparameters FOR REGRESSION
    """
    
    print("\n" + "="*80)
    print("RETRAINING WITH BEST HYPERPARAMETERS (REGRESSION)")
    print("="*80)
    
    # Build model
    model = FineTunedBERTaECFP(
        unfreeze_layers=best_params['unfreeze_layers'],
        num_heads=best_params['num_heads'],
        dropout=best_params['dropout'],
        hidden_dim=best_params['hidden_dim'],
        num_classifier_layers=best_params['num_classifier_layers'],
        fusion_type=best_params['fusion_type']
    ).to(DEVICE)
    
    # Datasets

    train_ds = BioActivityDataset(train_df, tokenizer, scaler=scaler, augment=True)
    val_ds = BioActivityDataset(val_df, tokenizer, scaler=train_ds.scaler, augment=False)
    test_ds = BioActivityDataset(test_df, tokenizer, scaler=scaler, augment=False)

    train_loader = DataLoader(train_ds, batch_size=best_params['batch_size'], 
                              shuffle=True)  
    val_loader = DataLoader(val_ds, batch_size=best_params['batch_size'], 
                            shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=best_params['batch_size'], 
                             shuffle=False)
    
    # ========================================================================
    # FIXED: Use REGRESSION loss
    # ========================================================================
    criterion = nn.MSELoss()  # ✅ Correct for regression
    # Alternative: nn.SmoothL1Loss() for robustness
    
    # Optimizer
    if best_params['optimizer'] == 'Adam':
        optimizer = optim.Adam(
            model.parameters(), 
            lr=best_params['learning_rate'], 
            weight_decay=best_params['weight_decay']
        )
    elif best_params['optimizer'] == 'RMSprop':
        optimizer = torch.optim.RMSprop(
            model.parameters(), 
            lr=best_params['learning_rate'], 
            weight_decay=best_params['weight_decay'], 
            momentum=0.9
        )
    else:
        optimizer = optim.AdamW(
            model.parameters(), 
            lr=best_params['learning_rate'], 
            weight_decay=best_params['weight_decay']
        )

    # Scheduler
    if best_params['scheduler'] == 'CosineAnnealingLR':
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    elif best_params['scheduler'] == 'ReduceLROnPlateau':
        # FIXED: mode='min' for regression
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.5, patience=3
        )
    elif best_params['scheduler'] == 'OneCycleLR':
        total_steps = len(train_loader) * epochs
        scheduler = optim.lr_scheduler.OneCycleLR(
            optimizer,
            max_lr=best_params['learning_rate'],
            total_steps=total_steps,
            pct_start=0.1,
            anneal_strategy='cos'
        )
    else:  # Lambda
        total_steps = len(train_loader) * epochs
        warmup_steps = int(0.1 * total_steps)
        scheduler = optim.lr_scheduler.LambdaLR(
            optimizer,
            lr_lambda=lambda step: min(step/warmup_steps, 1) *
                                   (0.5 * (1 + np.cos(np.pi * step / total_steps)))
        )

    # ========================================================================
    # Training loop with REGRESSION metrics
    # ========================================================================
    print("\nTraining final model...")
    best_rmse = float('inf')
    patience_counter = 0
    patience = 7
    
    scaler_amp = GradScaler()

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        
        for batch_idx, batch in enumerate(train_loader):
            logits = model(
                input_ids=batch['input_ids'].to(DEVICE),
                attention_mask=batch['attention_mask'].to(DEVICE),
                ecfp=batch['ecfp'].to(DEVICE),
                descriptors=batch['descriptors'].to(DEVICE)
            )
            
            labels = batch['labels'].float().to(DEVICE)
            
            # FIXED: MSE loss, no sigmoid
            loss = criterion(logits, labels) / best_params['grad_accum_steps']
            loss.backward()
                
            if (batch_idx + 1) % best_params['grad_accum_steps'] == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                optimizer.zero_grad()
            
            train_loss += loss.item() * best_params['grad_accum_steps']

            if best_params['scheduler'] in ['Lambda', 'OneCycleLR']:
                scheduler.step()

        avg_train_loss = train_loss / len(train_loader)
        
        # Step scheduler
        if best_params['scheduler'] == 'ReduceLROnPlateau':
            scheduler.step(avg_train_loss)
        elif best_params['scheduler'] == 'CosineAnnealingLR':
            scheduler.step()
        
        # ====================================================================
        # VALIDATION with REGRESSION metrics
        # ====================================================================
        model.eval()
        val_preds = []
        val_labels_list = []
        
        with torch.no_grad():
            for batch in val_loader:
                logits = model(
                    input_ids=batch['input_ids'].to(DEVICE),
                    attention_mask=batch['attention_mask'].to(DEVICE),
                    ecfp=batch['ecfp'].to(DEVICE),
                    descriptors=batch['descriptors'].to(DEVICE)
                )
                # NO SIGMOID for regression!
                val_preds.extend(logits.cpu().numpy())
                val_labels_list.extend(batch['labels'].cpu().numpy())
        
        val_preds = np.array(val_preds)
        val_labels_np = np.array(val_labels_list)
        
        val_rmse = np.sqrt(mean_squared_error(val_labels_np, val_preds))
        val_mae = mean_absolute_error(val_labels_np, val_preds)
        val_r2 = r2_score(val_labels_np, val_preds)
        
        print(f'Epoch {epoch+1:02d} | '
              f'TrainLoss {avg_train_loss:.4f} | '
              f'RMSE {val_rmse:.4f} | MAE {val_mae:.4f} | R² {val_r2:.4f}')
        
        # Early stopping
        if val_rmse < best_rmse:
            best_rmse = val_rmse
            patience_counter = 0
            torch.save(model.state_dict(), 'best_regression_model.pt')
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break
    
    return model

In [9]:
final_model = retrain_best_model(best_params, epochs=35)


RETRAINING WITH BEST HYPERPARAMETERS (REGRESSION)


Some weights of RobertaModel were not initialized from the model checkpoint at DeepChem/ChemBERTa-77M-MTR and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Computing ECFP...
Tokenizing SMILES...
Computing ECFP...
Tokenizing SMILES...
Computing ECFP...
Tokenizing SMILES...

Training final model...
Epoch 01 | TrainLoss 15.3978 | RMSE 2.2882 | MAE 1.9935 | R² -2.1498
Epoch 02 | TrainLoss 6.9076 | RMSE 1.9672 | MAE 1.7263 | R² -1.3281
Epoch 03 | TrainLoss 5.1288 | RMSE 1.7937 | MAE 1.5849 | R² -0.9356
Epoch 04 | TrainLoss 4.1669 | RMSE 1.6561 | MAE 1.4618 | R² -0.6500
Epoch 05 | TrainLoss 3.4767 | RMSE 1.5387 | MAE 1.3434 | R² -0.4244
Epoch 06 | TrainLoss 2.9477 | RMSE 1.4433 | MAE 1.2316 | R² -0.2533
Epoch 07 | TrainLoss 2.5171 | RMSE 1.3702 | MAE 1.1412 | R² -0.1295
Epoch 08 | TrainLoss 2.2075 | RMSE 1.3206 | MAE 1.0754 | R² -0.0492
Epoch 09 | TrainLoss 2.0214 | RMSE 1.2939 | MAE 1.0314 | R² -0.0072
Epoch 10 | TrainLoss 1.9253 | RMSE 1.2856 | MAE 1.0072 | R² 0.0057
Epoch 11 | TrainLoss 1.7726 | RMSE 1.2102 | MAE 0.9439 | R² 0.1190
Epoch 12 | TrainLoss 1.5409 | RMSE 1.0777 | MAE 0.8343 | R² 0.3013
Epoch 13 | TrainLoss 1.2886 | RMSE 0.9984 | 

In [ ]:
# After training completes
torch.save(final_model, 'best_regression_model_mlm.pth')

#### Train and evaluation

In [45]:
from sklearn.utils.class_weight import compute_class_weight
import torch.optim as optim

class_weights = compute_class_weight('balanced', classes=np.unique(train_df['bioactivity']), y=train_df['bioactivity'])
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

pos_weight = torch.tensor([978 / 4094]).to(device)  # From your imbalance
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = optim.AdamW(model.parameters(), lr=1e-5, weight_decay=0.01)  # Added weight decay
# scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)  # New: Better convergence
num_train_steps = len(train_loader) * 30
warmup_steps    = int(0.1 * num_train_steps)

scheduler = optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda=lambda step: min(step/warmup_steps, 1) * \
                           (0.5 * (1 + np.cos(np.pi * step / num_train_steps)))
)

# criterion = BCEWithLogitsLoss(pos_weight=class_weights[1])
epochs = 50  # Longer for fine-tuning
best_auc = 0
patience, counter = 7, 0

In [46]:
from sklearn.metrics import precision_recall_curve

for epoch in range(epochs):
    model.train()
    train_loss = 0
    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        descriptors = batch['descriptors'].to(device)
        labels = batch['labels'].float().to(device)
        ecfp = batch['ecfp'].to(device)
        
        optimizer.zero_grad()
        logits = model(input_ids=input_ids, attention_mask=attention_mask, ecfp=ecfp, descriptors=descriptors)
        loss = criterion(logits, labels)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # Gradient clipping
        optimizer.step()
        scheduler.step()  # Step LR scheduler per batch
        train_loss += loss.item()
    
    
    # Validation
    model.eval()
    val_preds, val_labels = [], []
    val_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            descriptors = batch['descriptors'].to(device)
            labels = batch['labels'].to(device)
            ecfp = batch['ecfp'].to(device)
            logits = model(input_ids, attention_mask, ecfp, descriptors)
            probs = torch.sigmoid(logits).cpu().numpy()
            val_preds.extend(probs)
            val_labels.extend(labels.cpu().numpy())
            loss = criterion(logits, labels.float())  # Compute loss for this batch
            val_loss += loss.item()  #
    
    
    auc = roc_auc_score(val_labels, val_preds)
    prec, rec, thr = precision_recall_curve(val_labels, val_preds)
    f1s = 2*prec*rec/(prec+rec+1e-8)
    best_thr = thr[np.argmax(f1s)]
    preds = (np.array(val_preds) > best_thr).astype(int)
    acc = accuracy_score(val_labels, preds)
    f1  = f1_score(val_labels, preds)

    print(f'Epoch {epoch+1:02d} | '
          f'TrainLoss {train_loss/len(train_loader):.4f} | '
          f'ValAUROC {auc:.4f} | Acc {acc:.4f} | F1 {f1:.4f} | Thr {best_thr:.3f}')
    
    if auc > best_auc:
        best_auc = auc
        torch.save(model, 'best_finetuned_model_tt.pth')
        counter = 0
    else:
        counter += 1
        if counter >= 7:
            print("Early stopping!")
            break

Epoch 01 | TrainLoss 0.2695 | ValAUROC 0.6842 | Acc 0.8328 | F1 0.9045 | Thr 0.471
Epoch 02 | TrainLoss 0.2544 | ValAUROC 0.8145 | Acc 0.8549 | F1 0.9171 | Thr 0.381
Epoch 03 | TrainLoss 0.2136 | ValAUROC 0.8761 | Acc 0.8801 | F1 0.9288 | Thr 0.177
Epoch 04 | TrainLoss 0.1705 | ValAUROC 0.9177 | Acc 0.8880 | F1 0.9324 | Thr 0.147
Epoch 05 | TrainLoss 0.1438 | ValAUROC 0.9418 | Acc 0.9132 | F1 0.9470 | Thr 0.213
Epoch 06 | TrainLoss 0.1209 | ValAUROC 0.9359 | Acc 0.9101 | F1 0.9448 | Thr 0.186
Epoch 07 | TrainLoss 0.1060 | ValAUROC 0.9454 | Acc 0.9196 | F1 0.9500 | Thr 0.184
Epoch 08 | TrainLoss 0.0954 | ValAUROC 0.9481 | Acc 0.9196 | F1 0.9500 | Thr 0.485
Epoch 09 | TrainLoss 0.0883 | ValAUROC 0.9503 | Acc 0.9274 | F1 0.9550 | Thr 0.181
Epoch 10 | TrainLoss 0.0788 | ValAUROC 0.9355 | Acc 0.9132 | F1 0.9473 | Thr 0.042
Epoch 11 | TrainLoss 0.0715 | ValAUROC 0.9487 | Acc 0.9290 | F1 0.9564 | Thr 0.222
Epoch 12 | TrainLoss 0.0692 | ValAUROC 0.9490 | Acc 0.9243 | F1 0.9529 | Thr 0.088
Epoc

#### Evaluation

In [10]:

model = FineTunedBERTaECFP(
        unfreeze_layers=best_params['unfreeze_layers'],
        num_heads=best_params['num_heads'],
        dropout=best_params['dropout'],
        hidden_dim=best_params['hidden_dim'],
        num_classifier_layers=best_params['num_classifier_layers'],
        fusion_type=best_params['fusion_type']
    ).to(DEVICE)
model.load_state_dict(torch.load('best_regression_model.pt'))
model.eval()    

FineTunedBERTaECFP(
  (bert): MolformerModel(
    (embeddings): MolformerEmbeddings(
      (word_embeddings): Embedding(2362, 768, padding_idx=2)
      (dropout): Dropout(p=0.2, inplace=False)
    )
    (encoder): MolformerEncoder(
      (layer): ModuleList(
        (0-11): 12 x MolformerLayer(
          (attention): MolformerAttention(
            (self): MolformerSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (rotary_embeddings): MolformerRotaryEmbedding()
              (feature_map): MolformerFeatureMap(
                (kernel): ReLU()
              )
            )
            (output): MolformerSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
              (d

In [10]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

test_ds = BioActivityDataset(test_df, tokenizer, scaler=scaler, augment=False)
val_ds = BioActivityDataset(val_df, tokenizer, scaler=scaler, augment=False)

test_loader = DataLoader(
    test_ds, 
    batch_size=32, 
    shuffle=False
)

val_loader = DataLoader(
    val_ds, 
    batch_size=32, 
    shuffle=False
)

# Run inference on test set
test_preds = []
test_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        descriptors = batch['descriptors'].to(DEVICE)
        ecfp = batch['ecfp'].to(DEVICE)
        labels = batch['labels'].to(DEVICE)
        
        logits = final_model(input_ids, attention_mask, ecfp, descriptors)
        
        # No sigmoid for regression - use raw predictions
        test_preds.extend(logits.cpu().numpy())
        test_labels.extend(labels.cpu().numpy())

# Convert to numpy arrays
test_preds = np.array(test_preds)
test_labels = np.array(test_labels)

# Calculate regression metrics
rmse = np.sqrt(mean_squared_error(test_labels, test_preds))
mae = mean_absolute_error(test_labels, test_preds)
r2 = r2_score(test_labels, test_preds)

print(f"Test RMSE: {rmse:.4f}")
print(f"Test MAE:  {mae:.4f}")
print(f"Test R²:   {r2:.4f}")

# Optional: Calculate percentage error
mape = np.mean(np.abs((test_labels - test_preds) / test_labels)) * 100
print(f"Test MAPE: {mape:.2f}%")

Computing ECFP...
Tokenizing SMILES...
Computing ECFP...
Tokenizing SMILES...
Test RMSE: 0.7965
Test MAE:  0.5961
Test R²:   0.6242
Test MAPE: 9.54%


In [19]:
# Step 1: Recreate the model architecture with same hyperparameters
loaded_model = FineTunedBERTaECFP(
    unfreeze_layers=best_params['unfreeze_layers'],
    num_heads=best_params['num_heads'],
    dropout=best_params['dropout'],
    hidden_dim=best_params['hidden_dim']
).to(DEVICE)

# Step 2: Load the saved weights
loaded_model.load_state_dict(torch.load('final_model_optuna.pt', map_location=DEVICE))

# Step 3: Set to evaluation mode
loaded_model.eval()

print("✅ Model loaded successfully!")

✅ Model loaded successfully!


In [ ]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score

test_preds = []
test_labels = []
test_probs = []
with torch.no_grad():
	for batch in test_loader:
		input_ids = batch['input_ids'].to(device)
		attention_mask = batch['attention_mask'].to(device)
		descriptors = batch['descriptors'].to(device)
		ecfp = batch['ecfp'].to(device)
		labels = batch['labels'].to(device)
		logits = loaded_model(input_ids, attention_mask, ecfp, descriptors)
		probs = torch.sigmoid(logits).cpu().numpy()
		test_probs.extend(probs)
		test_preds.extend((probs > 0.5).astype(int))
		test_labels.extend(labels.cpu().numpy())

accuracy = balanced_accuracy_score(test_labels, test_preds)
auc = roc_auc_score(test_labels, test_probs)
print(f"AUC: {auc}")
print(f"Accuracy: {accuracy}")
print("accuracy another: ", accuracy_score(test_labels, test_preds))

AUC: 0.9432102763385147
Accuracy: 0.8960276338514681
accuracy another:  0.9453781512605042


In [51]:
print(classification_report(test_labels, test_preds))

              precision    recall  f1-score   support

         0.0       0.84      0.88      0.86       180
         1.0       0.97      0.96      0.97       772

    accuracy                           0.95       952
   macro avg       0.91      0.92      0.91       952
weighted avg       0.95      0.95      0.95       952



In [52]:
# ...after training your model...
torch.save(final_model.state_dict(), "final_model.pt")

In [53]:
torch.save(final_model, "final_model_full.pth")

In [11]:
from sklearn.metrics import RocCurveDisplay
from sklearn.metrics import PrecisionRecallDisplay
import seaborn as sns
import matplotlib.pyplot as plt

def plot_confusion_matrix(y_test, y_pred):  
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Confusion Matrix')
    plt.show()


def plot_roc_curve(y_test, y_prob):
    RocCurveDisplay.from_predictions(y_test, y_prob)
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curve')
    plt.show()

def plot_auprc_curve(y_test, y_prob):
    PrecisionRecallDisplay.from_predictions(y_test, y_prob)
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title('AUPRC Curve')
    plt.show()


In [13]:
# Summary of Regression Metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, explained_variance_score, max_error, median_absolute_error

print("=" * 50)
print("REGRESSION METRICS SUMMARY")
print("=" * 50)

# Core metrics
rmse_test = np.sqrt(mean_squared_error(test_labels, test_preds))
mae_test = mean_absolute_error(test_labels, test_preds)
r2_test = r2_score(test_labels, test_preds)
mape_test = np.mean(np.abs((test_labels - test_preds) / (test_labels + 1e-8))) * 100
medae = median_absolute_error(test_labels, test_preds)
max_err = max_error(test_labels, test_preds)
explained_var = explained_variance_score(test_labels, test_preds)

print(f"Root Mean Squared Error (RMSE):    {rmse_test:.4f}")
print(f"Mean Absolute Error (MAE):         {mae_test:.4f}")
print(f"Median Absolute Error (MedAE):     {medae:.4f}")
print(f"R² Score:                          {r2_test:.4f}")
print(f"Mean Absolute Percentage Error:    {mape_test:.2f}%")
print(f"Max Error:                         {max_err:.4f}")
print(f"Explained Variance Score:          {explained_var:.4f}")
print("=" * 50)

# Error quantiles
print("\nError Quantiles:")
abs_errors = np.abs(test_labels - test_preds).flatten()
print(f"25th percentile: {np.percentile(abs_errors, 25):.4f}")
print(f"50th percentile (median): {np.percentile(abs_errors, 50):.4f}")
print(f"75th percentile: {np.percentile(abs_errors, 75):.4f}")
print(f"95th percentile: {np.percentile(abs_errors, 95):.4f}")
print("=" * 50)

REGRESSION METRICS SUMMARY
Root Mean Squared Error (RMSE):    0.7965
Mean Absolute Error (MAE):         0.5961
Median Absolute Error (MedAE):     0.4486
R² Score:                          0.6242
Mean Absolute Percentage Error:    9.54%
Max Error:                         3.5354
Explained Variance Score:          0.6279

Error Quantiles:
25th percentile: 0.2293
50th percentile (median): 0.4486
75th percentile: 0.8475
95th percentile: 1.5763


In [49]:
def shap_plot(final_model):
    import shap

    # Create a SHAP explainer
    #explainer = shap.TreeExplainer(final_model, X_train)
    explainer = shap.Explainer(final_model, train_df)

    # Calculate SHAP values for the test set
    shap_values = explainer.shap_values(test_df)

    # Plot summary plot
    #shap.summary_plot(shap_values, X_test, plot_type="bar")

    #emb_idx = slice(0, 1000)  # e.g., 0:1000
    #desc_idx = slice(1000, 1004)

    #print("Desc Index: " ,desc_idx)

    #explainer = shap.TreeExplainer(final_model)
    #shap_values = explainer.shap_values(X_test)  # Shape: [n_samples, n_features]

    # Overall importance
    #mean_abs_shap = np.abs(shap_values).mean(0)
    #print("Embedding total importance:", mean_abs_shap[emb_idx].sum())
    #print("Descriptors importance:", mean_abs_shap[desc_idx])

    plt.figure(figsize=(8,8))
    shap.summary_plot(shap_values, test_df)  # Clean plot!


In [50]:
shap_plot(final_model)

ValueError: Your currently installed version of Keras is Keras 3, but this is not yet supported in Transformers. Please install the backwards-compatible tf-keras package with `pip install tf-keras`.